# Capítulo 8 — Testes de comportamento

Este notebook reúne as funções da etapa06 e os mesmos oito testes de test_etapa06.py. Não usa os módulos de importação ou recorrência de capítulos posteriores. Execute as células em ordem e salve sua cópia.


## Definicoes


In [ ]:
"""Capítulo 6: prioridades, filtros e prazos, com dados em memória."""

from copy import deepcopy
from datetime import date


def cadastrar_tarefa(tarefas, titulo, prioridade="media", prazo=None):
    if not titulo.strip():
        return "Informe um título com pelo menos um caractere que não seja espaço."
    if prioridade not in ("baixa", "media", "alta"):
        return "Prioridade inválida. Use baixa, media ou alta."
    try:
        data_prazo = interpretar_prazo(prazo)
    except ValueError as erro:
        return str(erro)
    tarefas.append({
        "titulo": titulo,
        "concluida": False,
        "prioridade": prioridade,
        "prazo": data_prazo,
    })
    return "Tarefa cadastrada."


def concluir_tarefa(tarefas, indice):
    if indice < 0 or indice >= len(tarefas):
        return "Tarefa não encontrada."
    tarefas[indice]["concluida"] = True
    return "Tarefa concluída."


def listar_tarefas(tarefas):
    if not tarefas:
        return "Nenhuma tarefa cadastrada."
    linhas = []
    for indice, tarefa in enumerate(tarefas):
        marca = "x" if tarefa["concluida"] else " "
        linhas.append(f"{indice}: [{marca}] {tarefa['titulo']}")
    return "\n".join(linhas)


def filtrar_tarefas(tarefas, prioridade=None, concluida=None):
    if prioridade is not None and prioridade not in ("baixa", "media", "alta"):
        raise ValueError("Prioridade de filtro inválida.")
    if concluida is not None and type(concluida) is not bool:
        raise ValueError("Conclusão de filtro deve ser booleana.")
    resultado = []
    for tarefa in tarefas:
        if prioridade is not None and tarefa.get("prioridade", "media") != prioridade:
            continue
        if concluida is not None and tarefa["concluida"] != concluida:
            continue
        resultado.append(deepcopy(tarefa))
    return resultado


def interpretar_prazo(texto):
    if texto is None:
        return None
    if not isinstance(texto, str):
        raise ValueError("Prazo deve ser texto no formato AAAA-MM-DD.")
    try:
        prazo = date.fromisoformat(texto)
    except ValueError:
        raise ValueError("Prazo inválido. Use AAAA-MM-DD.") from None
    if prazo.isoformat() != texto:
        raise ValueError("Prazo inválido. Use AAAA-MM-DD.")
    return prazo


def esta_atrasada(tarefa, hoje):
    prazo = tarefa.get("prazo")
    return (
        not tarefa["concluida"]
        and prazo is not None
        and prazo < hoje
    )


## Demonstracao


In [ ]:
"""Oito testes dos recursos ensinados até o capítulo 8, sem módulos posteriores."""

import unittest
from copy import deepcopy
from datetime import date



class CadastroTests(unittest.TestCase):
    def test_titulo_vazio_preserva_lista(self):
        tarefas = [{"titulo": "Existente", "concluida": False}]
        antes = deepcopy(tarefas)
        mensagem = cadastrar_tarefa(tarefas, "", prioridade="alta")
        self.assertEqual(mensagem, "Informe um título com pelo menos um caractere que não seja espaço.")
        self.assertEqual(tarefas, antes)

    def test_prioridade_padrao_e_titulo_preservado(self):
        tarefas = []
        self.assertEqual(cadastrar_tarefa(tarefas, "  Ler  "), "Tarefa cadastrada.")
        self.assertEqual(tarefas, [{
            "titulo": "  Ler  ", "concluida": False, "prioridade": "media", "prazo": None,
        }])

    def test_recusas_e_ordem_das_validacoes(self):
        casos = [("", "urgente", "2026-02-30", "Informe")]
        casos += [("Válida", p, None, "Prioridade") for p in ("urgente", "Alta", " alta ", "")]
        casos += [("Válida", "alta", p, "Prazo") for p in ("", "2026-02-30", "20261014")]
        for titulo, prioridade, prazo, prefixo in casos:
            with self.subTest(titulo=titulo, prioridade=prioridade, prazo=prazo):
                tarefas = [{"titulo": "Existente", "concluida": False}]
                antes = deepcopy(tarefas)
                self.assertTrue(cadastrar_tarefa(tarefas, titulo, prioridade, prazo).startswith(prefixo))
                self.assertEqual(tarefas, antes)


class FiltrosTests(unittest.TestCase):
    def test_condicoes_ordem_e_copias(self):
        tarefas = [
            {"titulo": "Antiga", "concluida": False},
            {"titulo": "Alta aberta", "concluida": False, "prioridade": "alta"},
            {"titulo": "Alta concluída", "concluida": True, "prioridade": "alta"},
        ]
        antes = deepcopy(tarefas)
        self.assertEqual(len(filtrar_tarefas(tarefas, "alta")), 2)
        self.assertEqual(len(filtrar_tarefas(tarefas, concluida=False)), 2)
        resultado = filtrar_tarefas(tarefas, "alta", False)
        self.assertEqual([t["titulo"] for t in resultado], ["Alta aberta"])
        resultado[0]["titulo"] = "Alterado na cópia"
        self.assertEqual(tarefas, antes)
        self.assertEqual(filtrar_tarefas(tarefas, "media"), [tarefas[0]])
        self.assertEqual(filtrar_tarefas(tarefas), tarefas)
        with self.assertRaises(ValueError):
            filtrar_tarefas(tarefas, "urgente")
        with self.assertRaises(ValueError):
            filtrar_tarefas(tarefas, concluida="sim")


class PrazosTests(unittest.TestCase):
    def test_interpretacao_e_formato_exato(self):
        self.assertIsNone(interpretar_prazo(None))
        self.assertEqual(interpretar_prazo("2024-02-29"), date(2024, 2, 29))
        for entrada in ("2026-02-29", "20261014", "14/10/2026", "", 42):
            with self.subTest(entrada=entrada), self.assertRaises(ValueError):
                interpretar_prazo(entrada)

    def test_fronteiras_de_atraso(self):
        hoje = date(2026, 10, 14)
        for prazo, concluida, esperado in (
            (date(2026, 10, 13), False, True),
            (hoje, False, False),
            (date(2026, 10, 15), False, False),
            (date(2026, 10, 13), True, False),
            (None, False, False),
        ):
            with self.subTest(prazo=prazo, concluida=concluida):
                self.assertIs(esta_atrasada({"prazo": prazo, "concluida": concluida}, hoje), esperado)

    def test_cadastro_com_data_valida(self):
        tarefas = []
        self.assertEqual(cadastrar_tarefa(tarefas, "Revisar", "alta", "2026-10-14"),
                         "Tarefa cadastrada.")
        self.assertEqual(tarefas[0]["prazo"], date(2026, 10, 14))


class ApresentacaoTests(unittest.TestCase):
    def test_listagem_e_conclusao_preservadas(self):
        tarefas = [{"titulo": "Antiga", "concluida": False}]
        cadastrar_tarefa(tarefas, "Nova", "alta")
        self.assertEqual(listar_tarefas(tarefas), "0: [ ] Antiga\n1: [ ] Nova")
        for indice in (0, 1):
            concluir_tarefa(tarefas, indice)
            concluir_tarefa(tarefas, indice)
            self.assertIs(tarefas[indice]["concluida"], True)
        self.assertNotIn("prioridade", tarefas[0])
        self.assertEqual(tarefas[1]["prioridade"], "alta")
        antes = deepcopy(tarefas)
        self.assertEqual(concluir_tarefa(tarefas, 2), "Tarefa não encontrada.")
        self.assertEqual(tarefas, antes)
        self.assertEqual(listar_tarefas([]), "Nenhuma tarefa cadastrada.")




## Executar e investigar uma falha

A execução correta reúne oito testes. Para o experimento do capítulo, altere < para <= em esta_atrasada, execute novamente a célula de definições e esta célula de testes. O caso de vencimento no mesmo dia deve falhar. Restaure < e repita para concluir.


In [ ]:
suite = unittest.TestSuite()
for classe in (CadastroTests, FiltrosTests, PrazosTests, ApresentacaoTests):
    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(classe))
resultado = unittest.TextTestRunner(verbosity=2).run(suite)
assert resultado.wasSuccessful()
print("Oito testes executados sobre a etapa06.")
